# G14b0: does vLLM apply a LoRA adapter to the E4B? (canary adapter with random lora_B vs the base, same prompt, temperature 0)

In [ ]:
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try: os.unlink(pth)
        except OSError: pass
tmp = Path('/tmp/wheelhouse'); tmp.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    t = tmp / name
    if not t.exists(): os.symlink(w, t)
wheels = sorted(str(w) for w in tmp.glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('installed', len(wheels), 'wheels')

## Setup

In [ ]:
import json, os, random, shutil, sys, threading, time, traceback
from pathlib import Path
WORKING_DIR = Path('/kaggle/working'); WORKING_DIR.mkdir(parents=True, exist_ok=True)
G2 = WORKING_DIR / 'g14b0'; G2.mkdir(exist_ok=True); CODE = G2 / 'code'; CODE.mkdir(exist_ok=True)
RESULTS = {'started': time.strftime('%Y-%m-%d %H:%M:%S')}
import torch
_gpus = [(torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(torch.cuda.device_count())]
print('GPUs:', _gpus)
assert len(_gpus) == 4 and all(c[0] >= 8 for _, c in _gpus), f'expected 4x L4 (compute capability >= 8), got {_gpus}'
def save_results(): (G2 / 'g14b0_summary.json').write_text(json.dumps(RESULTS, indent=1, default=str))
BUDGET_MIN = 45; T_START = time.time()
def _watchdog():   # a hung round must not burn GPU quota
    RESULTS['watchdog'] = f'fired after {BUDGET_MIN} min'; save_results(); os._exit(3)
_wd = threading.Timer(BUDGET_MIN * 60, _watchdog); _wd.daemon = True; _wd.start()
for n, txt in {"capture_proxy.py": "\"\"\"Transparent logging reverse proxy between the harness (LiteLLM -> ADK) and vLLM, for G2 data collection.\n\nFor every non-streaming POST .../chat/completions it records, gzip-compressed, one JSON line with the FULL request body (messages incl. reasoning,\ntools, chat_template_kwargs, sampling params) and the FULL response body. It asks vLLM for `return_token_ids` so the record also carries the exact\nprompt token ids and completion token ids (no re-rendering guesses); if the server rejects that parameter it retries without it and keeps going.\nThe harness receives the response with the token-id fields stripped, i.e. exactly what it would get without the proxy.\nEverything else (GET /v1/models, /health, streaming requests) is passed through untouched.\n\"\"\"\n\nimport gzip\nimport hashlib\nimport json\nimport threading\nimport time\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\n\nimport requests\n\n\ndef task_key(messages: list[dict]) -> str:\n    \"\"\"Stable id of the agent session a request belongs to: hash of the system prompt + the first user message (the task statement).\"\"\"\n    sys_txt = next((m.get(\"content\") for m in messages if m.get(\"role\") == \"system\"), \"\") or \"\"\n    usr_txt = next((m.get(\"content\") for m in messages if m.get(\"role\") == \"user\"), \"\") or \"\"\n    h = hashlib.sha1()\n    for part in (sys_txt, usr_txt):\n        h.update((part if isinstance(part, str) else json.dumps(part, sort_keys=True)).encode(\"utf-8\", \"ignore\"))\n        h.update(b\"\\x00\")\n    return h.hexdigest()[:16]\n\n\nclass CaptureProxy:\n    def __init__(self, upstream: str, out_path: str, port: int = 8001, host: str = \"127.0.0.1\", ask_token_ids: bool = True):\n        self.upstream = upstream.rstrip(\"/\")          # e.g. http://127.0.0.1:8000\n        self.out_path = out_path\n        self.port, self.host = port, host\n        self.ask_token_ids = ask_token_ids\n        self.round_tag = \"r0\"\n        self.tokenize_fallback = False        # if True and vLLM returns no token ids, ask /tokenize for the exact prompt ids instead\n        self.n_records = 0\n        self.n_with_ids = 0\n        self.n_errors = 0\n        self._lock = threading.Lock()\n        self._httpd: ThreadingHTTPServer | None = None\n\n    # ---------------------------------------------------------------- lifecycle\n    def start(self) -> \"CaptureProxy\":\n        proxy = self\n\n        class H(BaseHTTPRequestHandler):\n            protocol_version = \"HTTP/1.1\"\n\n            def log_message(self, *a):  # silence\n                pass\n\n            def _relay(self, status: int, body: bytes, ctype: str = \"application/json\"):\n                self.send_response(status)\n                self.send_header(\"Content-Type\", ctype)\n                self.send_header(\"Content-Length\", str(len(body)))\n                self.end_headers()\n                self.wfile.write(body)\n\n            def do_GET(self):\n                try:\n                    r = requests.get(proxy.upstream + self.path, timeout=60)\n                    self._relay(r.status_code, r.content, r.headers.get(\"Content-Type\", \"application/json\"))\n                except Exception as e:  # noqa: BLE001\n                    self._relay(502, json.dumps({\"error\": str(e)}).encode())\n\n            def do_POST(self):\n                raw = self.rfile.read(int(self.headers.get(\"Content-Length\", \"0\")))\n                try:\n                    req = json.loads(raw) if raw else {}\n                except Exception:  # noqa: BLE001\n                    req = None\n                is_chat = self.path.rstrip(\"/\").endswith(\"/chat/completions\") and isinstance(req, dict) and not req.get(\"stream\")\n                if not is_chat:                      # pass-through (streaming, other endpoints)\n                    try:\n                        r = requests.post(proxy.upstream + self.path, data=raw, headers={\"Content-Type\": \"application/json\"}, timeout=1800)\n                        self._relay(r.status_code, r.content, r.headers.get(\"Content-Type\", \"application/json\"))\n                    except Exception as e:  # noqa: BLE001\n                        self._relay(502, json.dumps({\"error\": str(e)}).encode())\n                    return\n                t0 = time.time()\n                asked = proxy.ask_token_ids           # decided ONCE per request (another thread may flip the flag meanwhile)\n                body = dict(req, return_token_ids=True) if asked else req\n                try:\n                    r = requests.post(proxy.upstream + self.path, json=body, timeout=1800)\n                    if asked and r.status_code == 400 and \"return_token_ids\" in r.text:\n                        proxy.ask_token_ids = False   # this vLLM rejects the parameter: stop asking, redo this call plainly\n                        r = requests.post(proxy.upstream + self.path, json=req, timeout=1800)\n                except Exception as e:  # noqa: BLE001\n                    proxy.n_errors += 1\n                    self._relay(502, json.dumps({\"error\": f\"proxy upstream failure: {e}\"}).encode())\n                    return\n                try:\n                    resp = r.json()\n                except Exception:  # noqa: BLE001\n                    resp = None\n                if r.status_code == 200 and isinstance(resp, dict):\n                    proxy._record(req, resp, t0, time.time())\n                    clean = {k: v for k, v in resp.items() if k != \"prompt_token_ids\"}\n                    clean[\"choices\"] = [{k: v for k, v in ch.items() if k != \"token_ids\"} for ch in resp.get(\"choices\", [])]\n                    self._relay(200, json.dumps(clean).encode())\n                else:\n                    proxy.n_errors += 1\n                    self._relay(r.status_code, r.content, r.headers.get(\"Content-Type\", \"application/json\"))\n\n        self._httpd = ThreadingHTTPServer((self.host, self.port), H)\n        self._httpd.daemon_threads = True\n        threading.Thread(target=self._httpd.serve_forever, daemon=True).start()\n        return self\n\n    def stop(self) -> None:\n        if self._httpd:\n            self._httpd.shutdown()\n            self._httpd.server_close()\n\n    # ---------------------------------------------------------------- recording\n    def _record(self, req: dict, resp: dict, t0: float, t1: float) -> None:\n        has_ids = bool(resp.get(\"prompt_token_ids\")) and bool((resp.get(\"choices\") or [{}])[0].get(\"token_ids\"))\n        rec = {\"round\": self.round_tag, \"task_key\": task_key(req.get(\"messages\", [])), \"t0\": round(t0, 3), \"t1\": round(t1, 3), \"request\": req, \"response\": resp}\n        if not has_ids and self.tokenize_fallback:\n            try:\n                tk = {k: req[k] for k in (\"model\", \"messages\", \"tools\", \"chat_template_kwargs\") if k in req}\n                tk[\"add_generation_prompt\"] = True\n                r = requests.post(self.upstream + \"/tokenize\", json=tk, timeout=300)\n                if r.status_code == 200:\n                    rec[\"prompt_token_ids_tokenize\"] = r.json().get(\"tokens\")\n            except Exception:  # noqa: BLE001\n                pass\n        line = (json.dumps(rec, ensure_ascii=False) + \"\\n\").encode(\"utf-8\")\n        with self._lock:                             # one gzip member per record; the concatenation is a valid gzip stream\n            with open(self.out_path, \"ab\") as f:\n                f.write(gzip.compress(line, compresslevel=3))\n            self.n_records += 1\n            self.n_with_ids += has_ids\n\n    def stats(self) -> dict:\n        return {\"records\": self.n_records, \"with_token_ids\": self.n_with_ids, \"errors\": self.n_errors, \"asking_token_ids\": self.ask_token_ids, \"round\": self.round_tag}\n\n\ndef read_records(path: str):\n    \"\"\"Iterate the records of a capture file.\"\"\"\n    with gzip.open(path, \"rt\", encoding=\"utf-8\") as f:\n        for line in f:\n            if line.strip():\n                yield json.loads(line)\n"}.items(): (CODE / n).write_text(txt, encoding='utf-8')
sys.path.insert(0, str(CODE))
DATA_DIR = next(p.parent for p in Path('/kaggle/input').rglob('tasks.jsonl') if (p.parent / 'sample_submission').is_dir() and (p.parent / 'snapshots').is_dir())
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
arm_dirs = [p.parent for p in Path('/kaggle/input').rglob('agent.yaml') if 'swe_scaffold_v2_b1' in p.read_text(encoding='utf-8')]
assert len(arm_dirs) == 1, f'expected exactly one B1 arm, found {arm_dirs}'
ARM = WORKING_DIR / 'arm'; shutil.rmtree(ARM, ignore_errors=True); shutil.copytree(arm_dirs[0], ARM)
for _rel in ('agent.yaml', 'sub_agents/code_analyzer.yaml'):   # one variant for every agent and sub-agent
    _p = ARM / _rel; _t = _p.read_text(encoding='utf-8'); assert _t.count('gemma-4-31b-it-qat-w4a16-ct') == 1, _rel; _p.write_text(_t.replace('gemma-4-31b-it-qat-w4a16-ct', 'gemma-4-e4b-it-qat-w4a16-ct'), encoding='utf-8', newline='\n')
assert 'gemma-4-31b-it-qat-w4a16-ct' not in (ARM / 'agent.yaml').read_text()
(ARM / 'eval_config.yaml').write_text('# B2 (2026-10-20): budget candidate. 8 minutes / 28 tool calls (the harness states both in the task message), 150 s single-command timeout.\n# Why: public teams at 0.22-0.24 report 8 min / 28-30 calls (forum 750496) against our 5 min / no call cap (0.13-0.15); a tool-call budget also bounds the 12 h total:\n# 28 calls x ~9 s (sequential scorer, thinking on) ~ 4.2 min + overhead ~ 5 min per task ~ 10 h for ~120 tasks; exceeding 12 h errors the whole submission.\nevaluation:\n  timeout_seconds: 150\n  max_tool_calls: 28\n  max_time_minutes: 8\n', encoding='utf-8', newline='\n')   # B2 budget: 8 min / 28 tool calls / 150 s command timeout
(CODE / 'g3_wiring.py').write_text('"""Build the adapter evaluation arm and verify that a LoRA adapter really flows through the harness\'s compile path, BEFORE any GPU time is spent on training.\n\nmake_eval_arm: copy the B1 arm, add `adapter: <name>` right after the `model:` line of agent.yaml and sub_agents/code_analyzer.yaml, drop adapters/<name>/ in.\ncheck_wiring:  compile the arm exactly like swegemma 0.2.11 does (ModelRegistry with the base model and the adapter alias, discover_adapters manifest,\n               resolve_swegemma_adapter) and assert that the root agent\'s model resolves to the adapter alias (LiteLlm \'openai/<name>\').\n"""\n\nimport re\nimport shutil\nfrom pathlib import Path\n\nADAPTER = "g3_lora"\nBASE = "gemma-4-e4b-it-qat-w4a16-ct"\n\n\ndef make_eval_arm(b1_dir: Path, adapter_src: Path, dst: Path, name: str = ADAPTER) -> Path:\n    shutil.rmtree(dst, ignore_errors=True)\n    shutil.copytree(b1_dir, dst)\n    for rel in ("agent.yaml", "sub_agents/code_analyzer.yaml"):\n        p = dst / rel\n        s = p.read_text(encoding="utf-8")\n        pat = re.compile(rf"^(model: {re.escape(BASE)}\\s*\\n)", re.M)\n        assert len(pat.findall(s)) == 1, f"{rel}: expected exactly one `model: {BASE}` line"\n        assert "adapter:" not in s, f"{rel} already declares an adapter"\n        p.write_text(pat.sub(r"\\1" + f"adapter: {name}\\n", s, count=1), encoding="utf-8", newline="\\n")\n    (dst / "adapters" / name).mkdir(parents=True)\n    for f in ("adapter_config.json", "adapter_model.safetensors"):\n        shutil.copy(adapter_src / f, dst / "adapters" / name / f)\n    return dst\n\n\ndef check_wiring(arm: Path, name: str = ADAPTER) -> dict:\n    from adk_submission import ModelRegistry, compile_submission, discover_adapters, discover_declared_models\n    from google.adk.models.lite_llm import LiteLlm\n    from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS, build_submission_limits\n    from swegemma.context import SwegemmaContext\n    from swegemma.harness.agent_runner import resolve_swegemma_adapter\n    from swegemma.models import Task\n\n    task = Task(instance_id="d", repo="d/d", base_commit="0" * 40, problem_statement="d")\n    ctx = SwegemmaContext(docker_manager=None, container_id="", problem_statement="d", hints_text="", task=task, repo="d/d", graph_dir="/tmp", embeddings_dir="/tmp")\n    tools = ctx.create_tools()\n    declared = discover_declared_models(arm)\n    assert declared == {BASE}, declared\n    reg = ModelRegistry()\n    for m in declared:   # base model and adapter alias, as VllmServer.create_model_registry registers them when the server has an adapter manifest\n        reg.register(m, LiteLlm(model="openai/" + m, api_base="http://127.0.0.1:8000/v1", api_key="EMPTY"))\n    manifest = discover_adapters(str(arm), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)\n    assert set(manifest.adapters) == {name}, manifest.adapters\n    reg.register(name, LiteLlm(model="openai/" + name, api_base="http://127.0.0.1:8000/v1", api_key="EMPTY"))\n    limits, gen = build_submission_limits()\n    agent = compile_submission(submission_dir=arm, tool_registry=tools, model_registry=reg, limits=limits, generation_constraints=gen, adapter_manifest=manifest,\n                               adapter_resolver_fn=lambda b, i: resolve_swegemma_adapter(b, i, reg))\n    model = getattr(agent, "model", None)\n    model_id = getattr(model, "model", str(model))\n    assert model_id == f"openai/{name}", f"root agent resolves to {model_id!r}, expected \'openai/{name}\'"\n    return {"agent": agent.name, "root_model": model_id, "adapters": sorted(manifest.adapters)}\n', encoding='utf-8')
import g3_wiring as W
CANARY = next(p.parent for p in Path('/kaggle/input').rglob('adapter_model.safetensors') if 'canary' in str(p).lower()); print('canary adapter dir:', CANARY)
ARM = W.make_eval_arm(ARM, CANARY, WORKING_DIR / 'arm_canary'); RESULTS['wiring'] = W.check_wiring(ARM); print('wiring OK:', RESULTS['wiring'])
for must in ('agent.yaml', 'prompts/system.md', 'prompts/analyzer.md', 'configs/sampling.yaml', 'sub_agents/code_analyzer.yaml', 'eval_config.yaml'):
    assert (ARM / must).is_file(), f'arm missing {must}'
print('arm:', arm_dirs[0], '|', (ARM / 'configs/sampling.yaml').read_text().replace(chr(10), ' ')[-120:])
from swegemma.models import load_tasks
ALL_IDS = ["fastapi_11194", "fastapi_13537", "fastapi_13920", "fastapi_14077", "fastapi_14258", "fastapi_14301", "fastapi_14306", "fastapi_14372", "fastapi_14419", "fastapi_14448", "fastapi_14458", "fastapi_14463", "fastapi_14479", "fastapi_14487", "fastapi_14492", "fastapi_14616", "fastapi_14786", "fastapi_14794", "fastapi_14851", "fastapi_14873", "fastapi_14986", "fastapi_15280", "fastapi_15588", "fastapi_15589", "fastapi_15661", "fastapi_15800", "fastapi_5077", "fastapi_5624", "fastapi_9425", "fastapi_9555", "fastapi_9753", "requests_6644", "rich_2725", "rich_2943", "rich_3006", "rich_3043", "rich_3052", "rich_3061", "rich_3063", "rich_3067", "rich_3105", "rich_3180", "rich_3278", "rich_3454", "rich_3468", "rich_3470", "rich_3471", "rich_3472", "rich_3480", "rich_3506", "rich_3518", "rich_3521", "rich_3535", "rich_3675", "rich_3676", "rich_3718", "rich_3772", "rich_3777", "rich_3782", "rich_3882", "rich_3894", "rich_3905", "rich_3930", "rich_3934", "rich_3935", "rich_3938", "rich_3942", "rich_3944", "rich_3953", "rich_4006", "rich_4070", "rich_4075", "rich_4076", "rich_4077", "rich_4079"]
by_id = {t.instance_id: t for t in load_tasks(TASKS_PATH)}
missing = [i for i in ALL_IDS if i not in by_id]
assert not missing, f'task ids missing from the dataset: {missing}'
SCREEN_IDS = random.Random(77).sample(ALL_IDS, len(ALL_IDS))   # all 75 competition tasks, shuffled
assert len(SCREEN_IDS) == 75 and all(i in by_id for i in SCREEN_IDS)
SMOKE_IDS = random.Random(6).sample(ALL_IDS, 2)
print(len(ALL_IDS), 'tasks;', 'smoke ids:', SMOKE_IDS)

## vLLM (real 31B, no adapter) + capture proxy

In [ ]:
import litellm, torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-e4b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-e4b-it-qat-w4a16-ct/2')
assert MODEL_PATH.is_dir(), MODEL_PATH
first_arm = ARM
try:   # swegemma >= 0.2.11 / adk_submission >= 0.2.13 (wheelhouse of 2026-10-09)
    from adk_submission import discover_declared_models
    _declared = discover_declared_models(first_arm); assert len(_declared) == 1, f'expected exactly one declared model, got {_declared}'
    declared_model = next(iter(_declared))
except ImportError:   # older wheelhouse
    from swegemma.models.discovery import validate_single_declared_model
    declared_model = validate_single_declared_model(first_arm)
adapters = discover_adapters(str(first_arm), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
vllm_cfg = VllmConfig(model=str(MODEL_PATH), port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4',
    max_model_len=32768, dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90, enable_auto_tool_choice=True, enable_lora=True, max_loras=8, max_lora_rank=128,
    tensor_parallel_size=tp_size, startup_timeout=60 * 12)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')
import capture_proxy as CP
VLLM_URL = f'http://127.0.0.1:{server_instance.config.port}'
proxy = CP.CaptureProxy(VLLM_URL, str(G2 / 'capture.jsonl.gz'), port=8001).start()
server_instance.config.port = 8001      # the model registry reads base_url at creation time: from now on the harness talks to the capture proxy
models = server_instance.create_model_registry(aliases=[declared_model, TARGET_MODEL_NAME], model_prefix='openai/', api_key='EMPTY')
print('harness endpoint (proxy):', server_instance.base_url, '-> upstream', VLLM_URL)

## Smoke round, capture gate, full rounds

In [ ]:
import requests, math
base_name = str(MODEL_PATH); adapter_name = W.ADAPTER
ms = requests.get(VLLM_URL + '/v1/models', timeout=60).json()
ids = [m['id'] for m in ms['data']]; RESULTS['served_models'] = ids; print('served models:', ids)
assert adapter_name in ids, f'adapter {adapter_name} not registered: {ids}'
PROMPTS = ['Write a Python function that returns the n-th Fibonacci number, then explain it in one sentence.',
           'The bug: `rich.table.Table` drops the caption when `show_header=False`. Which file would you look at first, and why?']
def ask(model, prompt):
    body = {'model': model, 'messages': [{'role': 'user', 'content': prompt}], 'temperature': 0, 'max_tokens': 64, 'logprobs': True, 'top_logprobs': 1,
            'chat_template_kwargs': {'enable_thinking': False}}
    r = requests.post(VLLM_URL + '/v1/chat/completions', json=body, timeout=600); r.raise_for_status()
    ch = r.json()['choices'][0]
    return ch['message'].get('content') or '', [t['logprob'] for t in (ch.get('logprobs') or {}).get('content', [])]
out = []
for p in PROMPTS:
    tb, lb = ask(base_name, p); ta, la = ask(adapter_name, p); tb2, lb2 = ask(base_name, p)
    n = min(len(lb), len(la))
    mean_abs = sum(abs(a - b) for a, b in zip(lb[:n], la[:n])) / max(n, 1)
    out.append({'prompt': p[:60], 'base_text': tb[:160], 'adapter_text': ta[:160], 'text_differs': tb != ta, 'base_repeatable': tb == tb2, 'mean_abs_logprob_diff': round(mean_abs, 4)})
    print(json.dumps(out[-1], indent=1))
RESULTS['canary'] = out
effect = any(o['text_differs'] or o['mean_abs_logprob_diff'] > 0.01 for o in out)
RESULTS['canary_verdict'] = 'ADAPTER APPLIED (outputs differ from the base)' if effect else 'ADAPTER HAS NO EFFECT (identical to the base): do NOT train for E4B before this is understood'
print('VERDICT:', RESULTS['canary_verdict']); RESULTS['finished'] = time.strftime('%Y-%m-%d %H:%M:%S'); save_results(); _wd.cancel()
server_instance.stop() if hasattr(server_instance, 'stop') else None
